In [15]:
import pandas as pd
import numpy as np

In [16]:
input_file = "/events.csv"
events = pd.read_csv(input_file)
events.head()

,id_odsp,id_event,sort_order,time,text,event_type,event_type2,side,event_team,opponent,...,player_in,player_out,shot_place,shot_outcome,is_goal,location,bodypart,assist_method,situation,fast_break
0,UFot0hit/,UFot0hit1,1,2,Attempt missed. Mladen Petric (Hamburg) left f...,1,12.0,2,Hamburg SV,Borussia Dortmund,...,NaN,NaN,6.0,2.0,0,9.0,2.0,1,1.0,0
1,UFot0hit/,UFot0hit2,2,4,"Corner, Borussia Dortmund. Conceded by Dennis...",2,NaN,1,Borussia Dortmund,Hamburg SV,...,NaN,NaN,NaN,NaN,0,NaN,NaN,0,NaN,0
2,UFot0hit/,UFot0hit3,3,4,"Corner, Borussia Dortmund. Conceded by Heiko ...",2,NaN,1,Borussia Dortmund,Hamburg SV,...,NaN,NaN,NaN,NaN,0,NaN,NaN,0,NaN,0
3,UFot0hit/,UFot0hit4,4,7,Foul by Sven Bender (Borussia Dortmund).,3,NaN,1,Borussia Dortmund,Hamburg SV,...,NaN,NaN,NaN,NaN,0,NaN,NaN,0,NaN,0
4,UFot0hit/,UFot0hit5,5,7,Gokhan Tore (Hamburg) wins a free kick in the ...,8,NaN,2,Hamburg SV,Borussia Dortmund,...,NaN,NaN,NaN,NaN,0,2.0,NaN,0,NaN,0


# Event-Level Data Preparation

### I started from the raw event data by creating binary features for shot outcomes, locations, assist types, and play situations.

### Then I grouped the data by match (`id_odsp`) and team (`event_team`) to get summed statistics for each match-team combination.

### After that i created ratio features like shooting accuracy, conversion rate, and inside box ratio using safe division. This produced a clean, aggregated event-level dataset ready for match-level analysis.

In [27]:
events_prepared = events.copy()

events_prepared["is_shot_on_target"] = np.where(events_prepared["shot_outcome"] == 1, 1, 0)
events_prepared["SHOT_OFF_TARGET"] = np.where(events_prepared["shot_outcome"] == 2, 1, 0)
events_prepared["BLOCKED_SHOT"] = np.where(events_prepared["shot_outcome"] == 3, 1, 0)
events_prepared["HIT_THE_BAR"] = np.where(events_prepared["shot_outcome"] == 4, 1, 0)

events_prepared["is_inside_box"] = np.where(
    (events_prepared["location"] == 3) |
    (events_prepared["location"] == 9) |
    (events_prepared["location"] == 11) |
    (events_prepared["location"] == 13) |
    (events_prepared["location"] == 15),
    1,
    0
)

events_prepared["is_outside_box"] = np.where(events_prepared["location"] == 15, 1, 0)

events_prepared["is_long_range"] = np.where(
    (events_prepared["location"] == 16) |
    (events_prepared["location"] == 17) |
    (events_prepared["location"] == 18),
    1,
    0
)

events_prepared["PASS_ASSIST"] = np.where(events_prepared["assist_method"] == 1, 1, 0)
events_prepared["CROSS"] = np.where(events_prepared["assist_method"] == 2, 1, 0)
events_prepared["THROUGH_BALL"] = np.where(events_prepared["assist_method"] == 4, 1, 0)

events_prepared["OPEN_PLAY"] = np.where(events_prepared["situation"] == 1, 1, 0)
events_prepared["SET_PIECE"] = np.where(events_prepared["situation"] == 2, 1, 0)
events_prepared["CORNER"] = np.where(events_prepared["situation"] == 3, 1, 0)

events_prepared.head()

,id_odsp,id_event,sort_order,time,text,event_type,event_type2,side,event_team,opponent,...,HIT_THE_BAR,is_inside_box,is_outside_box,is_long_range,PASS_ASSIST,CROSS,THROUGH_BALL,OPEN_PLAY,SET_PIECE,CORNER
0,UFot0hit/,UFot0hit1,1,2,Attempt missed. Mladen Petric (Hamburg) left f...,1,12.0,2,Hamburg SV,Borussia Dortmund,...,0,1,0,0,1,0,0,1,0,0
1,UFot0hit/,UFot0hit2,2,4,"Corner, Borussia Dortmund. Conceded by Dennis...",2,NaN,1,Borussia Dortmund,Hamburg SV,...,0,0,0,0,0,0,0,0,0,0
2,UFot0hit/,UFot0hit3,3,4,"Corner, Borussia Dortmund. Conceded by Heiko ...",2,NaN,1,Borussia Dortmund,Hamburg SV,...,0,0,0,0,0,0,0,0,0,0
3,UFot0hit/,UFot0hit4,4,7,Foul by Sven Bender (Borussia Dortmund).,3,NaN,1,Borussia Dortmund,Hamburg SV,...,0,0,0,0,0,0,0,0,0,0
4,UFot0hit/,UFot0hit5,5,7,Gokhan Tore (Hamburg) wins a free kick in the ...,8,NaN,2,Hamburg SV,Borussia Dortmund,...,0,0,0,0,0,0,0,0,0,0


In [28]:
events_prepared = (
    events_prepared
    .groupby(["id_odsp", "event_team"], as_index=False)
    .agg(
        BLOCKED_SHOT_sum=("BLOCKED_SHOT", "sum"),
        SHOT_OFF_TARGET_sum=("SHOT_OFF_TARGET", "sum"),
        is_shot_on_target_sum=("is_shot_on_target", "sum"),
        is_goal_sum=("is_goal", "sum"),
        is_long_range_sum=("is_long_range", "sum"),
        is_outside_box_sum=("is_outside_box", "sum"),
        is_inside_box_sum=("is_inside_box", "sum"),
        THROUGH_BALL_sum=("THROUGH_BALL", "sum"),
        CROSS_sum=("CROSS", "sum"),
        PASS_ASSIST_sum=("PASS_ASSIST", "sum"),
        CORNER_sum=("CORNER", "sum"),
        SET_PIECE_sum=("SET_PIECE", "sum"),
        OPEN_PLAY_sum=("OPEN_PLAY", "sum"),
        fast_break_sum=("fast_break", "sum"),
        count=("id_odsp", "size")
    )
)

events_prepared.head()


,id_odsp,event_team,BLOCKED_SHOT_sum,SHOT_OFF_TARGET_sum,is_shot_on_target_sum,is_goal_sum,is_long_range_sum,is_outside_box_sum,is_inside_box_sum,THROUGH_BALL_sum,CROSS_sum,PASS_ASSIST_sum,CORNER_sum,SET_PIECE_sum,OPEN_PLAY_sum,fast_break_sum,count
0,004f4ING/,Southampton,7,4,4,0,0,6,14,0,4,4,0,0,15,0,46
1,004f4ING/,Swansea,2,2,1,1,0,4,6,0,1,4,0,1,5,0,31
2,00LMl81F/,AC Milan,3,8,4,3,1,7,12,0,4,8,2,0,13,2,56
3,00LMl81F/,AS Roma,8,7,9,2,1,8,21,1,4,11,3,0,20,0,69
4,00OX4xFp/,AS Monaco,1,5,3,0,0,4,8,1,5,1,1,0,8,0,50


In [29]:
events_prepared_prepared = events_prepared.copy()

#shooting accuracy = on target/(on target+off target)
events_prepared_prepared["Shooting_accuracy"] = np.where(
    (events_prepared_prepared["is_shot_on_target_sum"] +
     events_prepared_prepared["SHOT_OFF_TARGET_sum"]) > 0,
    events_prepared_prepared["is_shot_on_target_sum"] /
    (events_prepared_prepared["is_shot_on_target_sum"] +
     events_prepared_prepared["SHOT_OFF_TARGET_sum"]),
    0
)

#conversion rate = goals/shots on target
events_prepared_prepared["Conversion_rate"] = np.where(
    events_prepared_prepared["is_shot_on_target_sum"] > 0,
    events_prepared_prepared["is_goal_sum"] /
    events_prepared_prepared["is_shot_on_target_sum"],
    0
)

#inside box ratio = inside/(inside + outside)
events_prepared_prepared["Inside_box_ratio"] = np.where(
    (events_prepared_prepared["is_inside_box_sum"] +
     events_prepared_prepared["is_outside_box_sum"]) > 0,
    events_prepared_prepared["is_inside_box_sum"] /
    (events_prepared_prepared["is_inside_box_sum"] +
     events_prepared_prepared["is_outside_box_sum"]),
    0
)

events_prepared_prepared.head()

events_prepared_prepared.to_csv(
    "/events_prepared_prepared.csv",
    index=False
)

# prepare match-level dat
### we removed betting columns and created a match result label

In [32]:
match_level = pd.read_csv("/match-level.csv")

# Drop unnecessary betting columns
match_level_prepared = match_level.drop(columns=[
    "odd_over",
    "odd_bts",
    "odd_under",
    "odd_bts_n"
])

# Create result_actual column
match_level_prepared["result_actual"] = np.where(
    match_level_prepared["fthg"] > match_level_prepared["ftag"], "H",
    np.where(match_level_prepared["fthg"] == match_level_prepared["ftag"], "D", "A")
)

match_level_prepared.head()

match_level_prepared.to_csv("match_level_prepared.csv", index=False)

# MERGING MATCH-LEVEL WITH EVENT DATA

### Here we combine the event-level aggregated data. with the match-level dataset using id_odsp. we bring in match info like date, league, season, country, and team names (ht, at). after merging, we sort the data by date, match id, and event_team so everything is in proper order before moving into feature engineering

In [35]:
event_match = events_prepared_prepared.merge(
    match_level_prepared[
        ["id_odsp", "date", "league", "season", "country", "ht", "at"]
    ],
    on="id_odsp",
    how="left"
)

event_match.head()

event_match.to_csv("/event_match.csv", index=False)

In [36]:
event_match_sorted = event_match.sort_values(
    by=["date", "id_odsp", "event_team"]
).reset_index(drop=True)

event_match_sorted.head()

,id_odsp,event_team,BLOCKED_SHOT_sum,SHOT_OFF_TARGET_sum,is_shot_on_target_sum,is_goal_sum,is_long_range_sum,is_outside_box_sum,is_inside_box_sum,THROUGH_BALL_sum,...,count,Shooting_accuracy,Conversion_rate,Inside_box_ratio,date,league,season,country,ht,at
0,UFot0hit/,Borussia Dortmund,3,7,5,3,1,5,15,0,...,56,0.416667,0.600000,0.750000,2011-08-05,D1,2012,germany,Borussia Dortmund,Hamburg SV
1,UFot0hit/,Hamburg SV,4,2,2,1,1,2,7,0,...,54,0.500000,0.500000,0.777778,2011-08-05,D1,2012,germany,Borussia Dortmund,Hamburg SV
2,Aw5DflLH/,FC Augsburg,1,3,7,2,0,4,11,0,...,61,0.700000,0.285714,0.733333,2011-08-06,D1,2012,germany,FC Augsburg,SC Freiburg
3,Aw5DflLH/,SC Freiburg,3,2,4,2,0,5,7,0,...,61,0.666667,0.500000,0.583333,2011-08-06,D1,2012,germany,FC Augsburg,SC Freiburg
4,CzPV312a/,Lorient,2,2,4,1,0,5,7,0,...,56,0.666667,0.250000,0.583333,2011-08-06,F1,2012,france,Paris Saint-Germain,Lorient


In [37]:
event_match_sorted_prepared = event_match_sorted.copy()

event_match_sorted_prepared["is_home"] = np.where(
    event_match_sorted_prepared["event_team"] == event_match_sorted_prepared["ht"],
    1,
    0
)

event_match_sorted_prepared["is_away"] = np.where(
    event_match_sorted_prepared["event_team"] == event_match_sorted_prepared["at"],
    1,
    0
)

event_match_sorted_prepared.head()

,id_odsp,event_team,BLOCKED_SHOT_sum,SHOT_OFF_TARGET_sum,is_shot_on_target_sum,is_goal_sum,is_long_range_sum,is_outside_box_sum,is_inside_box_sum,THROUGH_BALL_sum,...,Conversion_rate,Inside_box_ratio,date,league,season,country,ht,at,is_home,is_away
0,UFot0hit/,Borussia Dortmund,3,7,5,3,1,5,15,0,...,0.600000,0.750000,2011-08-05,D1,2012,germany,Borussia Dortmund,Hamburg SV,1,0
1,UFot0hit/,Hamburg SV,4,2,2,1,1,2,7,0,...,0.500000,0.777778,2011-08-05,D1,2012,germany,Borussia Dortmund,Hamburg SV,0,1
2,Aw5DflLH/,FC Augsburg,1,3,7,2,0,4,11,0,...,0.285714,0.733333,2011-08-06,D1,2012,germany,FC Augsburg,SC Freiburg,1,0
3,Aw5DflLH/,SC Freiburg,3,2,4,2,0,5,7,0,...,0.500000,0.583333,2011-08-06,D1,2012,germany,FC Augsburg,SC Freiburg,0,1
4,CzPV312a/,Lorient,2,2,4,1,0,5,7,0,...,0.250000,0.583333,2011-08-06,F1,2012,france,Paris Saint-Germain,Lorient,0,1


In [38]:
home_matches = event_match_sorted_prepared[
    event_match_sorted_prepared["is_home"] == 1
].copy()

away_matches = event_match_sorted_prepared[
    event_match_sorted_prepared["is_away"] == 1
].copy()

In [39]:
home_matches_joined = home_matches.merge(
    away_matches,
    on="id_odsp",
    how="left",
    suffixes=("__home", "__away")
)

home_matches_joined.head()

,id_odsp,event_team__home,BLOCKED_SHOT_sum__home,SHOT_OFF_TARGET_sum__home,is_shot_on_target_sum__home,is_goal_sum__home,is_long_range_sum__home,is_outside_box_sum__home,is_inside_box_sum__home,THROUGH_BALL_sum__home,...,Conversion_rate__away,Inside_box_ratio__away,date__away,league__away,season__away,country__away,ht__away,at__away,is_home__away,is_away__away
0,UFot0hit/,Borussia Dortmund,3,7,5,3,1,5,15,0,...,0.500000,0.777778,2011-08-05,D1,2012,germany,Borussia Dortmund,Hamburg SV,0,1
1,Aw5DflLH/,FC Augsburg,1,3,7,2,0,4,11,0,...,0.500000,0.583333,2011-08-06,D1,2012,germany,FC Augsburg,SC Freiburg,0,1
2,CzPV312a/,Paris Saint-Germain,3,6,3,0,0,9,12,1,...,0.250000,0.583333,2011-08-06,F1,2012,france,Paris Saint-Germain,Lorient,0,1
3,GUOdmtII/,Caen,2,6,3,1,0,8,11,0,...,0.000000,0.750000,2011-08-06,F1,2012,france,Caen,Valenciennes,0,1
4,M7PhlM2C/,Brest,4,12,7,2,0,7,21,1,...,0.666667,0.692308,2011-08-06,F1,2012,france,Brest,Evian Thonon Gaillard,0,1


In [40]:
home_matches_joined_prepared = home_matches_joined.copy()

# Create match_result
home_matches_joined_prepared["match_result"] = np.where(
    home_matches_joined_prepared["is_goal_sum__home"] > home_matches_joined_prepared["is_goal_sum__away"], 1,
    np.where(
        home_matches_joined_prepared["is_goal_sum__home"] < home_matches_joined_prepared["is_goal_sum__away"], -1,
        0
    )
)

# Rename columns (clean up)
home_matches_joined_prepared = home_matches_joined_prepared.rename(columns={
    "date__home": "date",
    "season__home": "season",
    "league__home": "league",
    "country__home": "country",
    "id_odsp__home": "id_odsp"
})

home_matches_joined_prepared.head()

,id_odsp,event_team__home,BLOCKED_SHOT_sum__home,SHOT_OFF_TARGET_sum__home,is_shot_on_target_sum__home,is_goal_sum__home,is_long_range_sum__home,is_outside_box_sum__home,is_inside_box_sum__home,THROUGH_BALL_sum__home,...,Inside_box_ratio__away,date__away,league__away,season__away,country__away,ht__away,at__away,is_home__away,is_away__away,match_result
0,UFot0hit/,Borussia Dortmund,3,7,5,3,1,5,15,0,...,0.777778,2011-08-05,D1,2012,germany,Borussia Dortmund,Hamburg SV,0,1,1
1,Aw5DflLH/,FC Augsburg,1,3,7,2,0,4,11,0,...,0.583333,2011-08-06,D1,2012,germany,FC Augsburg,SC Freiburg,0,1,0
2,CzPV312a/,Paris Saint-Germain,3,6,3,0,0,9,12,1,...,0.583333,2011-08-06,F1,2012,france,Paris Saint-Germain,Lorient,0,1,-1
3,GUOdmtII/,Caen,2,6,3,1,0,8,11,0,...,0.750000,2011-08-06,F1,2012,france,Caen,Valenciennes,0,1,1
4,M7PhlM2C/,Brest,4,12,7,2,0,7,21,1,...,0.692308,2011-08-06,F1,2012,france,Brest,Evian Thonon Gaillard,0,1,0


# FEATURE ENGINEERING
### Here we split home/away, build rolling averages. for each team, then merge everything back together. and create features we’ll use for prediction

In [50]:
home_matches = event_match_sorted_prepared[
    event_match_sorted_prepared["is_home"] == 1
].copy()

away_matches = event_match_sorted_prepared[
    event_match_sorted_prepared["is_away"] == 1
].copy()

home_matches = home_matches.rename(columns={"event_team": "team"})
away_matches = away_matches.rename(columns={"event_team": "team"})

home_matches = home_matches.add_prefix("home__")
away_matches = away_matches.add_prefix("away__")

home_matches_joined = home_matches.merge(
    away_matches,
    left_on="home__id_odsp",
    right_on="away__id_odsp",
    how="left"
)

home_matches_joined = home_matches_joined.drop(columns=["away__id_odsp"])

home_matches_joined_prepared = home_matches_joined.copy()

home_matches_joined_prepared.columns = (
    home_matches_joined_prepared.columns
    .str.strip()
    .str.replace(" ", "_")
)

home_matches_joined_prepared["match_result"] = np.where(
    home_matches_joined_prepared["home__is_goal_sum"] > home_matches_joined_prepared["away__is_goal_sum"],
    1,
    np.where(
        home_matches_joined_prepared["home__is_goal_sum"] < home_matches_joined_prepared["away__is_goal_sum"],
        -1,
        0
    )
)

home_matches_joined_prepared = home_matches_joined_prepared.rename(columns={
    "home__date": "date",
    "home__season": "season",
    "home__league": "league",
    "home__country": "country",
    "home__id_odsp": "id_odsp",
    "home__team": "home_team",
    "away__team": "away_team"
})

home_matches_joined_prepared.to_csv("/home_matches_joined_prepared.csv", index=False)

home_matches_joined_prepared.head()

,id_odsp,home_team,home__BLOCKED_SHOT_sum,home__SHOT_OFF_TARGET_sum,home__is_shot_on_target_sum,home__is_goal_sum,home__is_long_range_sum,home__is_outside_box_sum,home__is_inside_box_sum,home__THROUGH_BALL_sum,...,away__Inside_box_ratio,away__date,away__league,away__season,away__country,away__ht,away__at,away__is_home,away__is_away,match_result
0,UFot0hit/,Borussia Dortmund,3,7,5,3,1,5,15,0,...,0.777778,2011-08-05,D1,2012,germany,Borussia Dortmund,Hamburg SV,0,1,1
1,Aw5DflLH/,FC Augsburg,1,3,7,2,0,4,11,0,...,0.583333,2011-08-06,D1,2012,germany,FC Augsburg,SC Freiburg,0,1,0
2,CzPV312a/,Paris Saint-Germain,3,6,3,0,0,9,12,1,...,0.583333,2011-08-06,F1,2012,france,Paris Saint-Germain,Lorient,0,1,-1
3,GUOdmtII/,Caen,2,6,3,1,0,8,11,0,...,0.750000,2011-08-06,F1,2012,france,Caen,Valenciennes,0,1,1
4,M7PhlM2C/,Brest,4,12,7,2,0,7,21,1,...,0.692308,2011-08-06,F1,2012,france,Brest,Evian Thonon Gaillard,0,1,0


In [52]:
data = home_matches_joined_prepared.copy()

data["date"] = pd.to_datetime(data["date"], errors="coerce")
data = data.reset_index(drop=True)
data["match_id"] = data.index
data = data.sort_values(["date", "match_id"]).reset_index(drop=True)

home = data[[
    "match_id", "date", "home_team", "away_team",
    "home__is_shot_on_target_sum", "home__Shooting_accuracy",
    "home__is_goal_sum", "home__is_inside_box_sum",
    "home__CROSS_sum", "home__PASS_ASSIST_sum",
    "home__CORNER_sum", "home__fast_break_sum"
]].copy()

home.columns = [
    "match_id", "date", "team", "opponent",
    "shots", "accuracy", "goals", "inside_box",
    "cross", "assist", "corner", "fast_break"
]

home["side"] = "home"

away = data[[
    "match_id", "date", "away_team", "home_team",
    "away__is_shot_on_target_sum", "away__Shooting_accuracy",
    "away__is_goal_sum", "away__is_inside_box_sum",
    "away__CROSS_sum", "away__PASS_ASSIST_sum",
    "away__CORNER_sum", "away__fast_break_sum"
]].copy()

away.columns = [
    "match_id", "date", "team", "opponent",
    "shots", "accuracy", "goals", "inside_box",
    "cross", "assist", "corner", "fast_break"
]

away["side"] = "away"

team_history = pd.concat([home, away], ignore_index=True)
team_history = team_history.sort_values(["team", "date", "match_id"]).reset_index(drop=True)

cols = ["shots", "accuracy", "goals", "inside_box", "cross", "assist", "corner", "fast_break"]

for col in cols:
    team_history[col + "_avg"] = (
        team_history.groupby("team")[col]
        .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
    )

home_avg = team_history[team_history["side"] == "home"][
    ["match_id"] + [col + "_avg" for col in cols]
].copy()

away_avg = team_history[team_history["side"] == "away"][
    ["match_id"] + [col + "_avg" for col in cols]
].copy()

home_avg = home_avg.rename(columns={
    "shots_avg": "home_shots_avg",
    "accuracy_avg": "home_accuracy_avg",
    "goals_avg": "home_goals_avg",
    "inside_box_avg": "home_inside_box_avg",
    "cross_avg": "home_cross_avg",
    "assist_avg": "home_assist_avg",
    "corner_avg": "home_corner_avg",
    "fast_break_avg": "home_fast_break_avg"
})

away_avg = away_avg.rename(columns={
    "shots_avg": "away_shots_avg",
    "accuracy_avg": "away_accuracy_avg",
    "goals_avg": "away_goals_avg",
    "inside_box_avg": "away_inside_box_avg",
    "cross_avg": "away_cross_avg",
    "assist_avg": "away_assist_avg",
    "corner_avg": "away_corner_avg",
    "fast_break_avg": "away_fast_break_avg"
})

cleaner_testing = data.merge(home_avg, on="match_id", how="left")
cleaner_testing = cleaner_testing.merge(away_avg, on="match_id", how="left")
cleaner_testing = cleaner_testing.drop(columns=["match_id"])

cleaner_testing.to_csv("/cleaner_testing.csv", index=False)

cleaner_testing.head()

,id_odsp,home_team,home__BLOCKED_SHOT_sum,home__SHOT_OFF_TARGET_sum,home__is_shot_on_target_sum,home__is_goal_sum,home__is_long_range_sum,home__is_outside_box_sum,home__is_inside_box_sum,home__THROUGH_BALL_sum,...,home_corner_avg,home_fast_break_avg,away_shots_avg,away_accuracy_avg,away_goals_avg,away_inside_box_avg,away_cross_avg,away_assist_avg,away_corner_avg,away_fast_break_avg
0,UFot0hit/,Borussia Dortmund,3,7,5,3,1,5,15,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Aw5DflLH/,FC Augsburg,1,3,7,2,0,4,11,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CzPV312a/,Paris Saint-Germain,3,6,3,0,0,9,12,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,GUOdmtII/,Caen,2,6,3,1,0,8,11,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,M7PhlM2C/,Brest,4,12,7,2,0,7,21,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [53]:
cleaning_testing2 = cleaner_testing.copy()

cleaning_testing2 = cleaning_testing2[
    cleaning_testing2["home_goals_avg"].notnull() &
    cleaning_testing2["away_goals_avg"].notnull()
].copy()

cols_to_drop = [
    col for col in cleaning_testing2.columns
    if ("_sum" in col and "_avg" not in col)
]

cleaning_testing2 = cleaning_testing2.drop(columns=cols_to_drop, errors="ignore")

extra_drop = [
    "home_Shooting_accuracy",
    "away_Shooting_accuracy",
    "home_Conversion_rate",
    "away_Conversion_rate",
    "home_Inside_box_ratio",
    "away_Inside_box_ratio",
    "home_count",
    "away_count"
]

cleaning_testing2 = cleaning_testing2.drop(columns=extra_drop, errors="ignore")

cleaning_testing2["goal_diff_avg"] = cleaning_testing2["home_goals_avg"] - cleaning_testing2["away_goals_avg"]
cleaning_testing2["shot_on_target_diff_avg"] = cleaning_testing2["home_shots_avg"] - cleaning_testing2["away_shots_avg"]
cleaning_testing2["shooting_accuracy_diff_avg"] = cleaning_testing2["home_accuracy_avg"] - cleaning_testing2["away_accuracy_avg"]
cleaning_testing2["inside_box_diff_avg"] = cleaning_testing2["home_inside_box_avg"] - cleaning_testing2["away_inside_box_avg"]
cleaning_testing2["cross_diff_avg"] = cleaning_testing2["home_cross_avg"] - cleaning_testing2["away_cross_avg"]
cleaning_testing2["pass_assist_diff_avg"] = cleaning_testing2["home_assist_avg"] - cleaning_testing2["away_assist_avg"]
cleaning_testing2["corner_diff_avg"] = cleaning_testing2["home_corner_avg"] - cleaning_testing2["away_corner_avg"]
cleaning_testing2["fast_break_diff_avg"] = cleaning_testing2["home_fast_break_avg"] - cleaning_testing2["away_fast_break_avg"]

keep_cols = [
    "date", "league", "season", "country", "home_team", "away_team", "match_result",
    "home_shots_avg", "home_accuracy_avg", "home_goals_avg", "home_inside_box_avg",
    "home_cross_avg", "home_assist_avg", "home_corner_avg", "home_fast_break_avg",
    "away_shots_avg", "away_accuracy_avg", "away_goals_avg", "away_inside_box_avg",
    "away_cross_avg", "away_assist_avg", "away_corner_avg", "away_fast_break_avg",
    "goal_diff_avg", "shot_on_target_diff_avg", "shooting_accuracy_diff_avg",
    "inside_box_diff_avg", "cross_diff_avg", "pass_assist_diff_avg",
    "corner_diff_avg", "fast_break_diff_avg"
]

cleaning_testing2 = cleaning_testing2[keep_cols].copy()

cleaning_testing2.to_csv("/cleaning_testing2.csv", index=False)

cleaning_testing2.head()

,date,league,season,country,home_team,away_team,match_result,home_shots_avg,home_accuracy_avg,home_goals_avg,...,away_corner_avg,away_fast_break_avg,goal_diff_avg,shot_on_target_diff_avg,shooting_accuracy_diff_avg,inside_box_diff_avg,cross_diff_avg,pass_assist_diff_avg,corner_diff_avg,fast_break_diff_avg
19,2011-08-13,D1,2012,germany,TSG Hoffenheim,Borussia Dortmund,1,5.0,0.625000,1.0,...,1.0,1.0,-2.0,0.0,0.208333,-5.0,1.0,-1.0,-1.0,-1.0
20,2011-08-13,F1,2012,france,St Etienne,AS Nancy Lorraine,1,2.0,0.250000,2.0,...,0.0,1.0,1.0,1.0,0.050000,4.0,1.0,2.0,1.0,-1.0
21,2011-08-13,D1,2012,germany,Nurnberg,Hannover 96,-1,7.0,0.636364,1.0,...,0.0,0.0,-1.0,3.0,-0.163636,6.0,1.0,3.0,1.0,0.0
22,2011-08-13,D1,2012,germany,Borussia Monchengladbach,VfB Stuttgart,0,3.0,0.750000,1.0,...,2.0,2.0,-2.0,-3.0,0.204545,-6.0,-3.0,0.0,-2.0,-2.0
23,2011-08-13,F1,2012,france,Valenciennes,Brest,0,6.0,0.666667,0.0,...,1.0,0.0,-2.0,-1.0,0.298246,-12.0,0.0,-8.0,0.0,0.0


# Splitting
### used earlier seasons for training and later seasons for testing

In [58]:
train = cleaning_testing2[cleaning_testing2["season"] < 2016].copy()
test= cleaning_testing2[cleaning_testing2["season"] >= 2016].copy()

train.to_csv("train.csv", index=False)
test.to_csv("test.csv", index=False)

print(train.shape, test.shape)
train.head()

(6231, 31) (2748, 31)


,date,league,season,country,home_team,away_team,match_result,home_shots_avg,home_accuracy_avg,home_goals_avg,...,away_corner_avg,away_fast_break_avg,goal_diff_avg,shot_on_target_diff_avg,shooting_accuracy_diff_avg,inside_box_diff_avg,cross_diff_avg,pass_assist_diff_avg,corner_diff_avg,fast_break_diff_avg
19,2011-08-13,D1,2012,germany,TSG Hoffenheim,Borussia Dortmund,1,5.0,0.625000,1.0,...,1.0,1.0,-2.0,0.0,0.208333,-5.0,1.0,-1.0,-1.0,-1.0
20,2011-08-13,F1,2012,france,St Etienne,AS Nancy Lorraine,1,2.0,0.250000,2.0,...,0.0,1.0,1.0,1.0,0.050000,4.0,1.0,2.0,1.0,-1.0
21,2011-08-13,D1,2012,germany,Nurnberg,Hannover 96,-1,7.0,0.636364,1.0,...,0.0,0.0,-1.0,3.0,-0.163636,6.0,1.0,3.0,1.0,0.0
22,2011-08-13,D1,2012,germany,Borussia Monchengladbach,VfB Stuttgart,0,3.0,0.750000,1.0,...,2.0,2.0,-2.0,-3.0,0.204545,-6.0,-3.0,0.0,-2.0,-2.0
23,2011-08-13,F1,2012,france,Valenciennes,Brest,0,6.0,0.666667,0.0,...,1.0,0.0,-2.0,-1.0,0.298246,-12.0,0.0,-8.0,0.0,0.0


In [60]:
test_prepared = test.drop(columns=["match_result"]).copy()

test_prepared.to_csv("test_prepared.csv", index=False)

test_prepared.head()

,date,league,season,country,home_team,away_team,home_shots_avg,home_accuracy_avg,home_goals_avg,home_inside_box_avg,...,away_corner_avg,away_fast_break_avg,goal_diff_avg,shot_on_target_diff_avg,shooting_accuracy_diff_avg,inside_box_diff_avg,cross_diff_avg,pass_assist_diff_avg,corner_diff_avg,fast_break_diff_avg
6312,2015-08-07,F1,2016,france,Lille,Paris Saint-Germain,5.0,0.433233,1.8,13.2,...,0.2,0.0,-1.4,-1.2,-0.167835,1.6,-0.8,1.4,1.2,0.2
6315,2015-08-08,E0,2016,england,Chelsea,Swansea,5.0,0.538889,1.6,13.2,...,0.8,0.0,0.0,0.2,-0.100303,3.2,-1.0,4.6,0.2,0.2
6316,2015-08-08,E0,2016,england,Leicester City,Sunderland,3.8,0.296667,2.2,13.0,...,0.6,0.2,1.2,-0.8,-0.190714,2.6,3.8,-2.6,2.0,-0.2
6317,2015-08-08,E0,2016,england,Manchester Utd,Tottenham,4.4,0.523810,0.6,14.0,...,1.2,0.0,-0.4,1.2,0.075595,1.8,-0.4,0.6,-0.8,0.0
6319,2015-08-08,F1,2016,france,Bastia,Stade Rennes,3.2,0.458095,1.0,8.6,...,0.6,0.4,0.6,0.2,0.084805,-0.4,0.4,-0.8,0.2,-0.4
